# RAG con Azure OpenAI + Amazon S3 Vectors

Pipeline completo:
1. **Crear índice** en S3 Vectors (bucket: `poc-chat-prospect-cib`, 1536 dims, cosine)
2. **ETL**: Extraer texto de PDFs/PPTX → Chunking → Embeddings con `text-embedding-3-small` → Indexar en S3 Vectors
3. **RAG**: Query con Azure OpenAI embeddings + S3 Vectors + GPT-5.1

### Modelos
| Componente | Modelo | Proveedor |
|------------|--------|-----------|
| Embeddings | `text-embedding-3-small` (1536 dims) | Azure OpenAI |
| LLM | GPT-5.1 | Azure OpenAI |
| Vector DB | Amazon S3 Vectors | AWS |

In [ ]:
import os
import time
import json
import boto3
import pandas as pd
from openai import AzureOpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from pptx import Presentation
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List, Dict, Any

load_dotenv()

## 1. Configuración de Clientes

In [ ]:
# Azure OpenAI
AZURE_ENDPOINT = "https://lau-prospectora-resource.cognitiveservices.azure.com/"
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_API_VERSION = "2024-12-01-preview"

# Deployments
EMBEDDING_DEPLOYMENT = "text-embedding-3-small-prospectora"
LLM_DEPLOYMENT = "gpt-5.1-prospectora"

# Cliente Azure OpenAI
azure_client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)

# S3 Vectors (Ohio)
s3_vectors_client = boto3.client("s3vectors", region_name="us-east-2")
S3_BUCKET_NAME = "poc-chat-prospect-cib"
S3_INDEX_NAME = "chat-cib-poc"  # Índice creado manualmente

print(f"Azure OpenAI endpoint: {AZURE_ENDPOINT}")
print(f"Embedding deployment: {EMBEDDING_DEPLOYMENT}")
print(f"LLM deployment: {LLM_DEPLOYMENT}")
print(f"S3 Vectors bucket: {S3_BUCKET_NAME}")

## 2. Crear Índice en S3 Vectors

Índice de 1536 dimensiones (text-embedding-3-small) con distancia coseno.

> **Ejecutar solo una vez.** Si el índice ya existe, esta celda dará error.

In [ ]:
# Crear el índice vectorial en S3 Vectors
try:
    response = s3_vectors_client.create_index(
        vectorBucketName=S3_BUCKET_NAME,
        indexName=S3_INDEX_NAME,
        dimension=1536,
        distanceMetric="cosine"
    )
    print(f"Índice '{S3_INDEX_NAME}' creado exitosamente.")
    print(f"ARN: {response.get('indexArn', 'N/A')}")
except Exception as e:
    if "already exists" in str(e).lower() or "ConflictException" in str(type(e).__name__):
        print(f"El índice '{S3_INDEX_NAME}' ya existe. Continuando...")
    else:
        raise e

## 3. ETL: Extraer Texto → Chunking → Embeddings → Indexar

Procesa PDFs y PPTX de la carpeta `data/`.

In [ ]:
def extract_text_from_pdf(path: str) -> str:
    """Extrae texto de un archivo PDF."""
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text


def extract_text_from_pptx(path: str) -> str:
    """Extrae texto de un archivo PowerPoint."""
    prs = Presentation(path)
    text = ""
    for slide in prs.slides:
        for shape in slide.shapes:
            if shape.has_text_frame:
                for paragraph in shape.text_frame.paragraphs:
                    line = paragraph.text.strip()
                    if line:
                        text += line + "\n"
        text += "\n"  # Separador entre slides
    return text


# Procesar todos los archivos de data/
data_dir = "./data"
all_text = ""

for filename in os.listdir(data_dir):
    filepath = os.path.join(data_dir, filename)
    
    if filename.lower().endswith(".pdf"):
        text = extract_text_from_pdf(filepath)
        print(f"PDF: {filename} → {len(text)} caracteres")
        all_text += text + "\n\n"
        
    elif filename.lower().endswith(".pptx"):
        text = extract_text_from_pptx(filepath)
        print(f"PPTX: {filename} → {len(text)} caracteres")
        all_text += text + "\n\n"
    else:
        print(f"SKIP: {filename} (formato no soportado)")

print(f"\nTotal: {len(all_text)} caracteres extraídos")

In [ ]:
# Chunking con RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(all_text)
print(f"Total chunks: {len(chunks)}")
print(f"Ejemplo chunk[0] ({len(chunks[0])} chars):\n{chunks[0][:200]}...")

In [ ]:
# Generar embeddings con Azure OpenAI text-embedding-3-small
def get_embedding_azure(text: str) -> List[float]:
    """Genera un embedding de 1536 dims con text-embedding-3-small via Azure OpenAI."""
    response = azure_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding


# Generar embeddings para todos los chunks
vectors = []
print(f"Generando embeddings para {len(chunks)} chunks...")

for i, chunk in enumerate(chunks):
    embedding = get_embedding_azure(chunk)
    vectors.append({
        "text": chunk,
        "embedding": embedding
    })
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(chunks)} embeddings generados...")

print(f"\n✅ {len(vectors)} embeddings generados (dim={len(vectors[0]['embedding'])})")

In [ ]:
# Subir vectores a S3 Vectors
s3_vectors_data = [
    {
        "key": f"chunk-{i}",
        "data": {
            "float32": item["embedding"]
        },
        "metadata": {
            "text": item["text"]
        }
    }
    for i, item in enumerate(vectors)
]

# S3 Vectors tiene un límite de vectores por llamada, subir en lotes de 50
BATCH_SIZE = 50
for start in range(0, len(s3_vectors_data), BATCH_SIZE):
    batch = s3_vectors_data[start:start + BATCH_SIZE]
    response = s3_vectors_client.put_vectors(
        vectorBucketName=S3_BUCKET_NAME,
        indexName=S3_INDEX_NAME,
        vectors=batch
    )
    print(f"Lote {start // BATCH_SIZE + 1}: {len(batch)} vectores subidos")

print(f"\n✅ {len(s3_vectors_data)} vectores indexados en S3 Vectors")

In [ ]:
# Verificar que los vectores se subieron correctamente
response = s3_vectors_client.list_vectors(
    vectorBucketName=S3_BUCKET_NAME,
    indexName=S3_INDEX_NAME
)

vector_count = len(response.get('vectors', []))
print(f"Vectores en el índice: {vector_count}")
for v in response.get('vectors', [])[:5]:
    print(f"  - {v['key']}")

## 4. RAG Directo con Métricas Aisladas

| Etapa | Modelo | Qué mide |
|-------|--------|---------|
| Embedding | text-embedding-3-small | Generación del vector de la query |
| Vector Search | S3 Vectors | Búsqueda vectorial exclusiva |
| LLM Generation | GPT-5.1 | Generación de respuesta |
| End-to-End | Todo | Pipeline completo |

In [ ]:
# Retriever desde S3 Vectors (recibe el vector ya generado)
def retrieve_from_s3vectors(query_vector: List[float], k: int = 5) -> List[Dict[str, Any]]:
    """Busca documentos en S3 Vectors. Usa 'vectors' (no 'matches') y 'distance' (no 'score')."""
    res = s3_vectors_client.query_vectors(
        vectorBucketName=S3_BUCKET_NAME,
        indexName=S3_INDEX_NAME,
        queryVector={"float32": query_vector},
        topK=k,
        returnMetadata=True,
        returnDistance=True
    )
    documents = []
    for match in res.get("vectors", []):
        documents.append({
            "text": match.get("metadata", {}).get("text", ""),
            "score": 1.0 - match.get("distance", 0.0)
        })
    return documents


# LLM call con GPT-5.1 via Azure OpenAI
def llm_call_azure(prompt: str, system_prompt: str = "") -> str:
    """Llama a GPT-5.1 en Azure OpenAI."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    response = azure_client.chat.completions.create(
        model=LLM_DEPLOYMENT,
        messages=messages,
        max_completion_tokens=1000,
        temperature=0.0
    )
    return response.choices[0].message.content


# RAG directo con métricas aisladas
def run_rag_azure(question: str, query_vector: List[float]) -> Dict[str, Any]:
    """Ejecuta el pipeline RAG: Search → LLM."""
    # Etapa 1: Vector Search
    t_search_start = time.perf_counter()
    docs = retrieve_from_s3vectors(query_vector)
    t_search_end = time.perf_counter()
    search_latency = (t_search_end - t_search_start) * 1000
    
    # Preparar contexto
    context = "\n\n".join([d["text"] for d in docs])
    
    # Etapa 2: LLM Generation
    system = (
        "Eres un asistente RAG formal de Cibertec. "
        "Responde a la pregunta del usuario utilizando únicamente el contexto proporcionado.\n"
        "Sé conciso, preciso y basa tu respuesta estrictamente en los hechos. "
        "Si la información no está en el contexto, indica amablemente que no cuentas con la información."
    )
    prompt = f"Contexto:\n{context}\n\nPregunta: {question}\n\nRespuesta:"
    
    t_llm_start = time.perf_counter()
    generation = llm_call_azure(prompt, system_prompt=system)
    t_llm_end = time.perf_counter()
    llm_latency = (t_llm_end - t_llm_start) * 1000
    
    return {
        "generation": generation,
        "search_latency": search_latency,
        "llm_latency": llm_latency,
        "documents_count": len(docs),
        "documents": docs
    }

In [ ]:
# Preguntas de prueba
test_questions = [
    "¿Qué convenios tiene Cibertec con la Policía Nacional del Perú (PNP)?",
    "¿Cuáles son los requisitos y el porcentaje de descuento de la Beca Socioeconómica?",
    "¿Qué beneficios de empleabilidad ofrece Cibertec a sus estudiantes y egresados?",
    "¿Qué convenios internacionales tiene Cibertec para la carrera de computación?",
    "¿Cuáles son los tipos de ingreso por modalidad en 2026?"
]

results = []

for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"[Pregunta {i+1}] {question}")
    print(f"{'='*60}")
    
    # Etapa 1: Embedding
    t_emb_start = time.perf_counter()
    query_vector = get_embedding_azure(question)
    t_emb_end = time.perf_counter()
    embedding_latency = (t_emb_end - t_emb_start) * 1000
    print(f"  Embedding Latency: {embedding_latency:.2f}ms")
    
    # Etapa 2 y 3: Search + LLM
    res = run_rag_azure(question, query_vector)
    e2e_latency = embedding_latency + res["search_latency"] + res["llm_latency"]
    
    print(f"  Search: {res['search_latency']:.2f}ms | LLM: {res['llm_latency']:.2f}ms | E2E: {e2e_latency:.2f}ms")
    print(f"  Docs recuperados: {res['documents_count']}")
    print(f"  Respuesta: {res['generation'][:150]}...")
    
    results.append({
        "Pregunta": question,
        "Respuesta": res["generation"],
        "Embedding Latency (ms)": embedding_latency,
        "Vector Search Latency (ms)": res["search_latency"],
        "LLM Generation Latency (ms)": res["llm_latency"],
        "End-to-End Latency (ms)": e2e_latency,
        "Documentos Recuperados": res["documents_count"]
    })

In [ ]:
# Tabla de resultados
df_results = pd.DataFrame(results)
df_results.to_csv("benchmark_azure_results.csv", index=False)

pd.set_option('display.max_columns', None)
display(df_results[[
    "Pregunta",
    "Embedding Latency (ms)",
    "Vector Search Latency (ms)",
    "LLM Generation Latency (ms)",
    "End-to-End Latency (ms)"
]])

In [ ]:
# Resumen promedio
print("\n" + "="*60)
print("BENCHMARK PROMEDIO: Azure OpenAI + S3 Vectors")
print("="*60)
print(f"Embedding (text-embedding-3-small): {df_results['Embedding Latency (ms)'].mean():.2f}ms")
print(f"Vector Search (S3 Vectors):         {df_results['Vector Search Latency (ms)'].mean():.2f}ms")
print(f"LLM Generation (GPT-5.1):           {df_results['LLM Generation Latency (ms)'].mean():.2f}ms")
print(f"End-to-End:                         {df_results['End-to-End Latency (ms)'].mean():.2f}ms")
print(f"\nModelos: text-embedding-3-small + GPT-5.1 (Azure OpenAI)")
print(f"Vector DB: S3 Vectors ({S3_BUCKET_NAME}/{S3_INDEX_NAME})")